### Info 
- Conda 3.10.18 환경에서 작업
- 데이터셋 생성시 전체를 모두 한번에 처리하지 않고 단계별로 전처리(트러블슈팅의 이유때문)
1. 원본 폴더 OriginDataset 과 변환작업 진행할 YoloV11Dataset을 준비
2. json -> txt 변환시 클래스는 파일명에서 질병코드를 따와서 분류하고 bbox는 json내의 좌표값사용
3. 일부 파일에 이미지 경계를 벗어나는  bbox 좌표값이 존재 -> 경계를 벗어나는 부분을 클리핑하여 처리
4. 기존 0~7까지 8가지 클래스를 제공하나, 해당 데이터 전처리시 0은 배 정상, 8은 사과 정상으로 클래스 1개 추가
#### 트러블슈팅
- 기존에는 단순히 Json에서 crop값(작물분류)으로 클래스를 , 좌표값으로 bbox를 구성하여 txt 변환진행했으나, 
- 일부 Json파일에 crop과 disease 코드가 일치하지 않는 파일을 발견
  - 예) 작물분류는 배 인데(원래는0~2까지) 질병코드가 4로 나오거나 사과인데(3~7) 질병코드가 1로 되어있는파일들이 존재
- 따라서 작물분류의경우 Json에 어노테이션을 참고하지 않고 파일명에 직접 드러난 질병코드로 클래스 분류하여 처리

# 1. 모듈 라이브러리 불러오기

In [2]:
#=================================
# 모듈 라이브러리 불러오기
#=================================
import os
import json
import shutil
from pathlib import Path
import yaml
import random
from tqdm import tqdm
import math
from PIL import Image

# 2.경로 설정 및 디렉토리 생성

In [4]:
#=====================================
# 경로 설정 및 디렉토리 생성
#=====================================

origin_root = Path("OriginDataset")
yolo_root = Path("YoloV11Dataset")

# YOLO 디렉토리 구조 생성
(yolo_root / "images" / "train").mkdir(parents=True, exist_ok=True)
(yolo_root / "images" / "val").mkdir(parents=True, exist_ok=True)
(yolo_root / "images" / "test").mkdir(parents=True, exist_ok=True)
(yolo_root / "labels" / "train").mkdir(parents=True, exist_ok=True)
(yolo_root / "labels" / "val").mkdir(parents=True, exist_ok=True)
(yolo_root / "labels" / "test").mkdir(parents=True, exist_ok=True)

# 클래스 매핑 정보
class_mapping = {
    ("01", 0): 0,  # 배 정상
    ("01", 1): 1,  # 배검은별무늬병
    ("01", 2): 2,  # 배과수화상병
    ("02", 0): 8,  # 사과 정상
    ("02", 3): 3,  # 사과갈색무늬병
    ("02", 4): 4,  # 사과과수화상병
    ("02", 5): 5,  # 사과부란병
    ("02", 6): 6,  # 사과점무늬낙엽병
    ("02", 7): 7   # 사과탄저병
}

# 3. Training 이미지 복사

In [5]:
#=====================================
# Training 이미지 복사
#=====================================
print("Training 이미지 복사 시작...")
source_dirs = [d for d in (origin_root / "Training").iterdir() 
               if d.is_dir() and d.name.startswith("[원천]")]

for src_dir in tqdm(source_dirs):
    for img_file in src_dir.glob("*"):
        if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png']:
            dest = yolo_root / "images" / "train" / img_file.name
            if not dest.exists():
                shutil.copy(img_file, dest)
print("Training 이미지 복사 완료!")

Training 이미지 복사 시작...


100%|██████████| 4/4 [00:00<00:00, 2584.69it/s]

Training 이미지 복사 완료!


# 4. JSON 라벨 변환

In [12]:
no_name_files = []

In [16]:
# 4번째 셀: JSON 라벨 변환 (Training → train, Validation → val/test 5:5 분할)

# 에러 로그를 저장할 딕셔너리
error_logs = {
    "parse_error": [],        # 파일명 파싱 실패
    "class_mapping_error": [], # 클래스 매핑 실패
    "json_load_error": [],     # JSON 로딩 실패
    "image_size_error": [],    # 이미지 크기 추출 실패
    "invalid_bbox": [],        # 유효하지 않은 바운딩 박스
    "image_not_found": []      # 이미지 파일 없음
}

def clean_filename(filename):
    """파일명에서 이미지 확장자 제거"""
    # 이미지 확장자 목록
    image_exts = ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']
    
    # 확장자 제거
    for ext in image_exts:
        if filename.endswith(ext):
            return filename[:-len(ext)]
    
    return filename

def parse_filename(json_path):
    """파일명에서 필요한 정보 추출"""
    # 파일명에서 .json 확장자 제거
    base_name = json_path.stem
    
    # 이미지 확장자 추가 제거
    base_name = clean_filename(base_name)
    
    parts = base_name.split('_')
    if len(parts) < 5:
        log_error("parse_error", f"파일명 파트 부족 ({len(parts)}개)", json_path.name)
        return None, None, base_name
    
    # 4번째 요소: 질병 코드 (00~07)
    disease_code = parts[3]
    # 5번째 요소: 과일 코드 (01: 배, 02: 사과)
    fruit_code = parts[4]
    
    try:
        disease_int = int(disease_code)
        return fruit_code, disease_int, base_name
    except ValueError:
        log_error("parse_error", f"질병 코드 변환 실패: '{disease_code}'", json_path.name)
        return None, None, base_name

def log_error(error_type, message, file_name):
    """에러 로그 기록"""
    error_logs[error_type].append((file_name, message))

def find_image(base_name, search_dir):
    """이미지 파일 찾기 (개선된 버전)"""
    # base_name에서 이미지 확장자 제거
    pure_name = clean_filename(base_name)
    
    # 가능한 모든 확장자로 검색
    for ext in ['.jpg', '.JPG', '.png', '.PNG', '.jpeg', '.JPEG']:
        # 순수 파일명 + 확장자
        img_path = search_dir / f"{pure_name}{ext}"
        if img_path.exists():
            return img_path
        
        # 원본 base_name + 확장자
        img_path2 = search_dir / f"{base_name}{ext}"
        if img_path2.exists():
            return img_path2
    
    # 파일명에 공백이 있는 경우 대체
    if ' ' in pure_name:
        pure_name = pure_name.replace(' ', '_')
        for ext in ['.jpg', '.JPG', '.png', '.PNG', '.jpeg', '.JPEG']:
            img_path = search_dir / f"{pure_name}{ext}"
            if img_path.exists():
                return img_path
    
    # 마지막 시도: 대소문자 무시 검색
    for file in search_dir.iterdir():
        if file.stem.lower() == pure_name.lower() and file.suffix.lower() in ['.jpg', '.jpeg', '.png']:
            return file
    
    return None

def convert_json(json_path, image_path, txt_path):
    """JSON 파일을 YOLO 형식으로 변환"""
    # 파일명에서 정보 추출
    fruit_code, disease_int, base_name = parse_filename(json_path)
    if fruit_code is None or disease_int is None:
        return False
    
    # 매핑에서 클래스 ID 찾기
    class_id = None
    
    # 배(01)인 경우
    if fruit_code == "01":
        if disease_int == 0:  # 배 정상
            class_id = 0
        elif disease_int == 1:  # 배검은별무늬병
            class_id = 1
        elif disease_int == 2:  # 배과수화상병
            class_id = 2
    
    # 사과(02)인 경우
    elif fruit_code == "02":
        if disease_int == 0:  # 사과 정상
            class_id = 8
        elif disease_int == 3:  # 사과갈색무늬병
            class_id = 3
        elif disease_int == 4:  # 사과과수화상병
            class_id = 4
        elif disease_int == 5:  # 사과부란병
            class_id = 5
        elif disease_int == 6:  # 사과점무늬낙엽병
            class_id = 6
        elif disease_int == 7:  # 사과탄저병
            class_id = 7
    
    if class_id is None:
        log_error("class_mapping_error", 
                 f"과일:{fruit_code}, 질병:{disease_int}", 
                 json_path.name)
        return False
    
    # JSON 데이터 로드
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        log_error("json_load_error", str(e), json_path.name)
        return False
    
    # 이미지 크기 가져오기 (JSON에서 직접 추출)
    try:
        img_width = data["description"]["width"]
        img_height = data["description"]["height"]
    except KeyError:
        # JSON에 정보가 없으면 이미지 파일에서 추출
        try:
            if image_path:
                with Image.open(image_path) as img:
                    img_width, img_height = img.size
            else:
                log_error("image_size_error", 
                         "이미지 경로 없고 JSON에 크기 정보 없음", 
                         json_path.name)
                return False
        except Exception as e:
            log_error("image_size_error", str(e), json_path.name)
            return False
    
    # 어노테이션 처리 - points 배열에서 bbox 추출
    lines = []
    annotations = data.get("annotations", {})
    points = annotations.get("points", [])
    
    # points가 배열인 경우
    if isinstance(points, list) and points:
        for point in points:
            # bbox 좌표 추출 (xtl, ytl, xbr, ybr)
            xtl = point.get("xtl", 0)
            ytl = point.get("ytl", 0)
            xbr = point.get("xbr", 0)
            ybr = point.get("ybr", 0)
            
            # 좌표 유효성 검사 및 보정
            xtl = max(0, min(xtl, img_width - 1))
            ytl = max(0, min(ytl, img_height - 1))
            xbr = max(0, min(xbr, img_width - 1))
            ybr = max(0, min(ybr, img_height - 1))
            
            # bbox 크기 계산
            w = xbr - xtl
            h = ybr - ytl
            
            # 너비나 높이가 0이면 건너뜀
            if w <= 0 or h <= 0:
                continue
            
            # YOLO 형식으로 변환 (정규화된 중심좌표)
            x_center = (xtl + w / 2) / img_width
            y_center = (ytl + h / 2) / img_height
            w_norm = w / img_width
            h_norm = h / img_height
            
            # 값이 0~1 사이인지 확인
            if (0 <= x_center <= 1 and 0 <= y_center <= 1 and 
                0 <= w_norm <= 1 and 0 <= h_norm <= 1):
                lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")
            else:
                # 경계를 벗어나는 경우 보정된 값을 사용
                x_center = max(0.0, min(1.0, x_center))
                y_center = max(0.0, min(1.0, y_center))
                w_norm = max(0.0, min(1.0, w_norm))
                h_norm = max(0.0, min(1.0, h_norm))
                lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")
                log_error("invalid_bbox", 
                         f"x:{x_center}, y:{y_center}, w:{w_norm}, h:{h_norm} (보정됨)", 
                         json_path.name)
    
    # TXT 파일 저장 (객체가 없어도 파일 생성)
    with open(txt_path, 'w') as f:
        f.write("\n".join(lines))
    
    return True

# Training 라벨 처리
print("Training 라벨 처리 시작...")
train_label_dirs = [d for d in (origin_root / "Training").iterdir() 
                   if d.is_dir() and d.name.startswith("[라벨]")]

for label_dir in train_label_dirs:
    print(f"> 폴더 처리 중: {label_dir.name}")
    
    # 이미지 폴더 경로 생성
    image_dir_name = label_dir.name.replace("[라벨]", "[원천]")
    source_dir = origin_root / "Training" / image_dir_name
    
    # 폴더 존재 여부 확인
    if not source_dir.exists():
        print(f"  경고: 이미지 폴더가 존재하지 않습니다: {source_dir}")
        print("  대체 경로 탐색 시도...")
        # 대체 경로: 폴더명에서 공백 제거
        image_dir_name_alt = image_dir_name.replace(" ", "")
        source_dir_alt = origin_root / "Training" / image_dir_name_alt
        if source_dir_alt.exists():
            print(f"  대체 폴더 사용: {source_dir_alt}")
            source_dir = source_dir_alt
        else:
            print(f"  오류: 대체 폴더도 존재하지 않습니다. 건너뜁니다.")
            continue
    
    json_files = list(label_dir.glob("*.json"))
    for json_path in tqdm(json_files, desc=label_dir.name):
        # 파일명 정제: JSON 확장자 제거 + 이미지 확장자 제거
        clean_name = clean_filename(json_path.stem)
        txt_path = yolo_root / "labels" / "train" / f"{clean_name}.txt"
        
        # 이미 존재하면 건너뜀
        if txt_path.exists():
            continue
            
        # 이미지 파일 찾기 (정제된 파일명 사용)
        image_path = find_image(clean_name, source_dir)
        
        if not image_path:
            log_error("image_not_found", 
                     f"폴더: {source_dir.name}", 
                     clean_name)
            # 이미지 없어도 TXT 파일 생성 (JSON 데이터 사용)
            success = convert_json(json_path, None, txt_path)
            if not success:
                # 변환 실패시 생성된 TXT 파일 삭제
                if txt_path.exists():
                    txt_path.unlink()
            continue
            
        # JSON 변환 수행
        success = convert_json(json_path, image_path, txt_path)
        if not success:
            # 실패한 경우 생성된 TXT 파일 삭제
            if txt_path.exists():
                txt_path.unlink()

# Validation 라벨 처리 (val test 5:5 분할)
print("\nValidation 라벨 처리 시작...")
val_label_dirs = [d for d in (origin_root / "Validation").iterdir() 
                 if d.is_dir() and d.name.startswith("[라벨]")]

all_val_jsons = []
for label_dir in val_label_dirs:
    print(f"> 폴더 스캔 중: {label_dir.name}")
    all_val_jsons.extend(list(label_dir.glob("*.json")))

# 5:5 분할 (무작위 셔플)
random.seed(42)  # 재현성을 위한 시드 고정
random.shuffle(all_val_jsons)
split_idx = len(all_val_jsons) // 2
val_jsons = all_val_jsons[:split_idx]
test_jsons = all_val_jsons[split_idx:]

print(f"Validation 데이터 분할: val={len(val_jsons)}, test={len(test_jsons)}")

def process_val_jsons(json_list, target_dir):
    """Validation JSON 처리 함수"""
    for json_path in tqdm(json_list, desc=f"Processing {target_dir}"):
        # 파일명 정제: JSON 확장자 제거 + 이미지 확장자 제거
        clean_name = clean_filename(json_path.stem)
        txt_path = yolo_root / "labels" / target_dir / f"{clean_name}.txt"
        
        if txt_path.exists():
            continue
            
        # 이미지 폴더 경로 생성
        image_dir_name = json_path.parent.name.replace("[라벨]", "[원천]")
        source_dir = origin_root / "Validation" / image_dir_name
        
        # 폴더 존재 여부 확인
        if not source_dir.exists():
            # 대체 경로: 폴더명에서 공백 제거
            image_dir_name_alt = image_dir_name.replace(" ", "")
            source_dir_alt = origin_root / "Validation" / image_dir_name_alt
            if source_dir_alt.exists():
                source_dir = source_dir_alt
        
        # 이미지 파일 찾기 (정제된 파일명 사용)
        image_path = find_image(clean_name, source_dir)
        
        if not image_path:
            log_error("image_not_found", 
                     f"폴더: {source_dir.name}", 
                     clean_name)
            # 이미지 없어도 TXT 파일 생성 (JSON 데이터 사용)
            success = convert_json(json_path, None, txt_path)
            if not success:
                # 변환 실패시 생성된 TXT 파일 삭제
                if txt_path.exists():
                    txt_path.unlink()
            continue
            
        success = convert_json(json_path, image_path, txt_path)
        if not success:
            # 실패한 경우 생성된 TXT 파일 삭제
            if txt_path.exists():
                txt_path.unlink()

# val 및 test 처리
process_val_jsons(val_jsons, "val")
process_val_jsons(test_jsons, "test")
print("라벨 변환 완료!")

# 에러 리포트 출력
print("\n" + "="*50)
print("에러 리포트 요약")
print("="*50)

total_errors = 0
for error_type, errors in error_logs.items():
    if errors:
        print(f"\n[{error_type.upper()}]: {len(errors)}건")
        # 각 에러 유형별로 최대 5개 샘플 출력
        for i, (file_name, message) in enumerate(errors[:5]):
            print(f"  {i+1}. {file_name}: {message}")
        if len(errors) > 5:
            print(f"  ... (총 {len(errors)}건, 상위 5건만 출력)")
        total_errors += len(errors)

print("\n" + "="*50)
print(f"총 에러 건수: {total_errors}")
print("="*50)

Training 라벨 처리 시작...
> 폴더 처리 중: [라벨]배_0.정상


[라벨]배_0.정상: 100%|██████████| 20434/20434 [00:33<00:00, 607.92it/s]


> 폴더 처리 중: [라벨]배_1.질병


[라벨]배_1.질병: 100%|██████████| 2557/2557 [00:03<00:00, 640.66it/s]


> 폴더 처리 중: [라벨]사과_0.정상


...

> 폴더 처리 중: [라벨]사과_1.질병


[라벨]사과_1.질병: 100%|██████████| 8286/8286 [00:16<00:00, 500.71it/s]



Validation 라벨 처리 시작...
> 폴더 스캔 중: [라벨]배_0.정상
> 폴더 스캔 중: [라벨]배_1.질병
> 폴더 스캔 중: [라벨]사과_0.정상
> 폴더 스캔 중: [라벨]사과_1.질병
Validation 데이터 분할: val=3755, test=3756


Processing test: 100%|██████████| 3756/3756 [00:37<00:00, 100.78it/s]

라벨 변환 완료!

에러 리포트 요약

[IMAGE_NOT_FOUND]: 60015건
  1. V006_80_0_00_01_01_25_0_b06_20201005_0001_S01_1: 폴더: [원천]배_0.정상
  2. V006_80_0_00_01_01_25_0_b06_20201005_0004_S01_1: 폴더: [원천]배_0.정상
  3. V006_80_0_00_01_01_25_0_b06_20201005_0005_S01_1: 폴더: [원천]배_0.정상
  4. V006_80_0_00_01_01_25_0_b06_20201005_0007_S01_1: 폴더: [원천]배_0.정상
  5. V006_80_0_00_01_01_25_0_b06_20201005_0008_S01_1: 폴더: [원천]배_0.정상
  ... (총 60015건, 상위 5건만 출력)

총 에러 건수: 60015


# 5. val/test 이미지 복사

In [18]:
# val/test 이미지 복사 (수정된 버전)
print("val/test 이미지 복사 시작...")

def copy_val_images(label_dir, img_target):
    """라벨에 대응하는 이미지 복사"""
    # 원본 Validation 폴더의 [원천] 디렉토리 목록 가져오기
    source_dirs = [d for d in (origin_root / "Validation").iterdir() 
                 if d.is_dir() and d.name.startswith("[원천]")]
    
    for txt_path in tqdm(list(label_dir.glob("*.txt")), desc=f"Copying to {img_target}"):
        base_name = txt_path.stem
        
        # 이미지 검색
        image_path = None
        for source_dir in source_dirs:
            img_path = find_image(base_name, source_dir)
            if img_path:
                image_path = img_path
                break
        
        if image_path:
            # 이미지 확장자 유지하여 복사
            dest = yolo_root / "images" / img_target / image_path.name
            if not dest.exists():
                shutil.copy(image_path, dest)
        else:
            print(f"  이미지 파일 없음: {base_name}")

copy_val_images(yolo_root / "labels" / "val", "val")
copy_val_images(yolo_root / "labels" / "test", "test")
print("이미지 복사 완료!")

val/test 이미지 복사 시작...


Copying to test: 100%|██████████| 3756/3756 [01:37<00:00, 38.53it/s] 

이미지 복사 완료!


# 7. 폴더 검증 및 YAML 생성

In [19]:
# 폴더 검증 및 YAML 생성
print("\n데이터셋 검증 시작...")

def verify_dataset():
    """데이터셋 무결성 검증"""
    # 각 세트별 검증
    for set_name in ["train", "val", "test"]:
        img_dir = yolo_root / "images" / set_name
        label_dir = yolo_root / "labels" / set_name
        
        # 이미지와 라벨 파일 목록 (확장자 제외)
        images = {f.stem for f in img_dir.iterdir() if f.is_file()}
        labels = {f.stem for f in label_dir.iterdir() if f.suffix == ".txt"}
        
        # 교집검 계산
        common = images & labels
        img_only = images - labels
        label_only = labels - images
        
        print(f"\n[{set_name.upper()} SET]")
        print(f"이미지: {len(images)}개, 라벨: {len(labels)}개")
        print(f"일치 파일: {len(common)}개")
        print(f"이미지만 있는 파일: {len(img_only)}개")
        print(f"라벨만 있는 파일: {len(label_only)}개")
        
        # 문제 파일 샘플 출력
        if img_only:
            print(f"  이미지 샘플: {list(img_only)[:3]}")
        if label_only:
            print(f"  라벨 샘플: {list(label_only)[:3]}")

# 전체 검증 수행
verify_dataset()

# dataset.yaml 생성
yaml_data = {
    "path": str(yolo_root),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {
        0: '배 정상',
        1: '배검은별무늬병',
        2: '배과수화상병',
        3: '사과갈색무늬병',
        4: '사과과수화상병',
        5: '사과부란병',
        6: '사과점무늬낙엽병',
        7: '사과탄저병',
        8: '사과 정상'
    }
}

with open(yolo_root / "dataset.yaml", "w") as f:
    yaml.dump(yaml_data, f, sort_keys=False, allow_unicode=True)

print("\n검증 완료 및 dataset.yaml 생성!")


데이터셋 검증 시작...

[TRAIN SET]
이미지: 4개, 라벨: 60015개
일치 파일: 0개
이미지만 있는 파일: 4개
라벨만 있는 파일: 60015개
  이미지 샘플: ['V006_80_0_00_02_01_25_0_a01_20201028_0010_S01', 'V006_80_0_00_01_01_25_0_b06_20201005_0011_S01_1', 'V006_80_1_01_01_01_23_1_1483y_20201029_272']
  라벨 샘플: ['V006_80_0_00_01_05_25_0_c06_20201026_0135_S01_1', 'V006_80_0_00_02_04_26_0_c37_201202_0448_S01_1', 'V006_80_0_00_01_05_25_0_c06_20201029_0029_S01_1']

[VAL SET]
이미지: 3755개, 라벨: 3755개
일치 파일: 3755개
이미지만 있는 파일: 0개
라벨만 있는 파일: 0개

[TEST SET]
이미지: 3756개, 라벨: 3756개
일치 파일: 3756개
이미지만 있는 파일: 0개
라벨만 있는 파일: 0개

검증 완료 및 dataset.yaml 생성!
